In [1]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import timm
import matplotlib.pyplot as plt
import random
import shutil
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {device}")
print(f"✅ GPU    : {torch.cuda.get_device_name(0)}")

# Create folders
BASE_DIR = "E:/document_forensics/module5_aidetection/module5b_aiface"
folders  = [
    "dataset/real",
    "dataset/fake",
    "dataset/train/real",
    "dataset/train/fake",
    "dataset/val/real",
    "dataset/val/fake",
    "dataset/test/real",
    "dataset/test/fake",
    "models",
    "results"
]
for f in folders:
    os.makedirs(os.path.join(BASE_DIR, f), exist_ok=True)

print("✅ Folders created!")

✅ Device : cuda
✅ GPU    : NVIDIA GeForce RTX 4070 Ti SUPER
✅ Folders created!


In [4]:
import shutil, os
from tqdm import tqdm

DEEPFAKE_ROOT = "E:/DEEPFAKE/real_vs_fake/real-vs-fake"
BASE_DIR      = "E:/document_forensics/module5_aidetection/module5b_aiface"

real_count = 0
fake_count = 0

def copy_images(src_dir, dst_dir, prefix, count):
    os.makedirs(dst_dir, exist_ok=True)
    files = [f for f in os.listdir(src_dir)
             if f.lower().endswith(('.jpg','.jpeg','.png'))]
    for f in tqdm(files, desc=f"Copying {prefix}"):
        src  = os.path.join(src_dir, f)
        dst  = os.path.join(dst_dir, f"{prefix}_{count:06d}.jpg")
        img  = cv2.imread(src)
        if img is None:
            continue
        img  = cv2.resize(img, (224, 224))
        cv2.imwrite(dst, img)
        count += 1
    return count

# Copy from train, test, valid
for split in ["train", "test", "valid"]:
    real_src = os.path.join(DEEPFAKE_ROOT, split, "real")
    fake_src = os.path.join(DEEPFAKE_ROOT, split, "fake")

    if os.path.exists(real_src):
        real_count = copy_images(real_src,
                        f"{BASE_DIR}/dataset/real",
                        "real", real_count)
    if os.path.exists(fake_src):
        fake_count = copy_images(fake_src,
                        f"{BASE_DIR}/dataset/fake",
                        "fake", fake_count)

print(f"\n✅ Real faces : {real_count:,}")
print(f"✅ Fake faces : {fake_count:,}")
print(f"✅ Total      : {real_count + fake_count:,}")

Copying fake: 100%|██████████| 10000/10000 [00:16<00:00, 595.87it/s]


✅ Real faces : 70,000
✅ Fake faces : 70,000
✅ Total      : 140,000


In [3]:
import os

DEEPFAKE_ROOT = "E:/DEEPFAKE/real_vs_fake/real-vs-fake"

total = 0
for split in ["train", "test", "valid"]:
    for label in ["real", "fake"]:
        path  = os.path.join(DEEPFAKE_ROOT, split, label)
        if os.path.exists(path):
            count = len(os.listdir(path))
            total += count
            print(f"  {split}/{label}: {count:,}")

print(f"\n✅ TOTAL images in dataset: {total:,}")

  train/real: 50,000
  train/fake: 50,000
  test/real: 10,000
  test/fake: 10,000
  valid/real: 10,000
  valid/fake: 10,000

✅ TOTAL images in dataset: 140,000


In [5]:
BASE_DIR = "E:/document_forensics/module5_aidetection/module5b_aiface"
REAL_DIR = f"{BASE_DIR}/dataset/real"
FAKE_DIR = f"{BASE_DIR}/dataset/fake"

real_imgs = os.listdir(REAL_DIR)
fake_imgs = os.listdir(FAKE_DIR)

random.shuffle(real_imgs)
random.shuffle(fake_imgs)

def split_list(lst):
    n = len(lst)
    return lst[:int(n*0.8)], lst[int(n*0.8):int(n*0.9)], lst[int(n*0.9):]

real_train, real_val, real_test = split_list(real_imgs)
fake_train, fake_val, fake_test = split_list(fake_imgs)

def copy_files(files, src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    for f in tqdm(files, desc=f"-> {os.path.basename(dst_dir)}"):
        shutil.copy2(os.path.join(src_dir, f),
                     os.path.join(dst_dir, f))

print("Splitting dataset...")
copy_files(real_train, REAL_DIR, f"{BASE_DIR}/dataset/train/real")
copy_files(real_val,   REAL_DIR, f"{BASE_DIR}/dataset/val/real")
copy_files(real_test,  REAL_DIR, f"{BASE_DIR}/dataset/test/real")
copy_files(fake_train, FAKE_DIR, f"{BASE_DIR}/dataset/train/fake")
copy_files(fake_val,   FAKE_DIR, f"{BASE_DIR}/dataset/val/fake")
copy_files(fake_test,  FAKE_DIR, f"{BASE_DIR}/dataset/test/fake")

print(f"\n✅ Train -> Real: {len(real_train):,} | Fake: {len(fake_train):,}")
print(f"✅ Val   -> Real: {len(real_val):,}   | Fake: {len(fake_val):,}")
print(f"✅ Test  -> Real: {len(real_test):,}  | Fake: {len(fake_test):,}")

Splitting dataset...


-> fake: 100%|██████████| 7000/7000 [00:03<00:00, 2142.85it/s]


✅ Train -> Real: 56,000 | Fake: 56,000
✅ Val   -> Real: 7,000   | Fake: 7,000
✅ Test  -> Real: 7,000  | Fake: 7,000


In [6]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os, random, torch

class FaceDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.transform = transform
        self.images    = []
        self.labels    = []

        for f in os.listdir(real_dir):
            self.images.append(os.path.join(real_dir, f))
            self.labels.append(0)  # 0 = Real

        for f in os.listdir(fake_dir):
            self.images.append(os.path.join(fake_dir, f))
            self.labels.append(1)  # 1 = Fake

        combined = list(zip(self.images, self.labels))
        random.shuffle(combined)
        self.images, self.labels = zip(*combined)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.images[idx]).convert("RGB")
        except:
            img = Image.new("RGB", (224, 224))
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

BASE_DIR = "E:/document_forensics/module5_aidetection/module5b_aiface/dataset"

train_dataset = FaceDataset(f"{BASE_DIR}/train/real", f"{BASE_DIR}/train/fake", train_transform)
val_dataset   = FaceDataset(f"{BASE_DIR}/val/real",   f"{BASE_DIR}/val/fake",   val_transform)
test_dataset  = FaceDataset(f"{BASE_DIR}/test/real",  f"{BASE_DIR}/test/fake",  val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print(f"✅ Train samples : {len(train_dataset):,}")
print(f"✅ Val samples   : {len(val_dataset):,}")
print(f"✅ Test samples  : {len(test_dataset):,}")
print(f"✅ Train batches : {len(train_loader):,}")
print(f"🚀 DataLoaders ready!")

✅ Train samples : 112,000
✅ Val samples   : 14,000
✅ Test samples  : 14,000
✅ Train batches : 1,750
🚀 DataLoaders ready!


In [7]:
import timm
import torch.nn as nn

class AIFaceDetector(nn.Module):
    def __init__(self):
        super(AIFaceDetector, self).__init__()
        self.model = timm.create_model(
            'efficientnet_b3',
            pretrained=True,
            num_classes=2
        )
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(self.model.classifier.in_features, 2)
        )

    def forward(self, x):
        return self.model(x)

model     = AIFaceDetector().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model      : EfficientNetB3")
print(f"✅ Parameters : {total_params:,}")
print(f"✅ Device     : {device}")
print(f"🚀 AI Face Detector ready!")

✅ Model      : EfficientNetB3
✅ Parameters : 10,699,306
✅ Device     : cuda
🚀 AI Face Detector ready!


In [8]:
EPOCHS    = 15
BEST_ACC  = 0.0
MODEL_DIR = "E:/document_forensics/module5_aidetection/module5b_aiface/models"

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

print("🚀 Training AI Face Detector!")
print(f"📊 Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")
print("=" * 60)

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        _, predicted   = outputs.max(1)
        train_total   += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  "):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs      = model(imgs)
            loss         = criterion(outputs, labels)

            val_loss    += loss.item()
            _, predicted = outputs.max(1)
            val_total   += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    t_loss = train_loss / len(train_loader)
    v_loss = val_loss   / len(val_loader)
    t_acc  = 100. * train_correct / train_total
    v_acc  = 100. * val_correct   / val_total

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    train_accs.append(t_acc)
    val_accs.append(v_acc)

    scheduler.step()

    if v_acc > BEST_ACC:
        BEST_ACC = v_acc
        torch.save(model.state_dict(),
                   f"{MODEL_DIR}/best_aiface_detector.pth")
        saved = "💾 Saved Best!"
    else:
        saved = ""

    print(f"\nEpoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.2f}% | "
          f"Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.2f}% {saved}")
    print("-" * 60)

print(f"\n🏆 Best Val Accuracy: {BEST_ACC:.2f}%")

🚀 Training AI Face Detector!
📊 Train: 112,000 | Val: 14,000


Epoch 1/15 [Val]  : 100%|██████████| 219/219 [00:38<00:00,  5.71it/s]



Epoch 01/15 | Train Loss: 0.0742 | Train Acc: 97.07% | Val Loss: 0.0126 | Val Acc: 99.59% 💾 Saved Best!
------------------------------------------------------------


Epoch 2/15 [Val]  : 100%|██████████| 219/219 [00:37<00:00,  5.90it/s]



Epoch 02/15 | Train Loss: 0.0238 | Train Acc: 99.13% | Val Loss: 0.0113 | Val Acc: 99.56% 
------------------------------------------------------------


...


Epoch 08/15 | Train Loss: 0.0045 | Train Acc: 99.86% | Val Loss: 0.0037 | Val Acc: 99.90% 💾 Saved Best!
------------------------------------------------------------


Epoch 9/15 [Val]  : 100%|██████████| 219/219 [00:39<00:00,  5.55it/s]



Epoch 09/15 | Train Loss: 0.0033 | Train Acc: 99.89% | Val Loss: 0.0017 | Val Acc: 99.96% 💾 Saved Best!
------------------------------------------------------------


Epoch 10/15 [Train]:  62%|██████▏   | 1085/1750 [07:00<04:17,  2.58it/s]


KeyboardInterrupt: 

In [9]:
MODEL_DIR = "E:/document_forensics/module5_aidetection/module5b_aiface/models"
model.load_state_dict(torch.load(
    f"{MODEL_DIR}/best_aiface_detector.pth",
    map_location=device, weights_only=True))
model.eval()

test_correct = 0
test_total   = 0

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Testing"):
        imgs, labels = imgs.to(device), labels.to(device)
        outputs      = model(imgs)
        _, predicted = outputs.max(1)
        test_total  += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_acc = 100. * test_correct / test_total
print(f"\n{'='*45}")
print(f"🏆 FINAL TEST ACCURACY : {test_acc:.2f}%")
print(f"✅ Total test images   : {test_total:,}")
print(f"✅ Correct             : {test_correct:,}")
print(f"❌ Wrong               : {test_total - test_correct:,}")
print(f"{'='*45}")

Testing: 100%|██████████| 219/219 [00:39<00:00,  5.57it/s]


🏆 FINAL TEST ACCURACY : 99.93%
✅ Total test images   : 14,000
✅ Correct             : 13,990
❌ Wrong               : 10
